# CyberLoRA 一键训练（Colab T4 16GB）

把 20~30 张本人照片训练成专属人物 LoRA（`cyberboy_sdxl.safetensors`）。

**修复过的 6 个致命 bug（务必照做）：**
1. 训练器用 `sdxl_train_network.py`（不是 `train_network.py`）。
2. 直接克隆 `kohya-ss/sd-scripts`（tag v0.8.7），不要用 `bmaltais/kohya_ss`（子模块无克隆为空）。
3. 用 `accelerate launch` 启动，并先 `write_basic_config()`（单卡非交互）。
4. 必开 `--gradient_checkpointing`（T4 16GB 否则 OOM）。
5. 必开 `--no_half_vae`（避免 SDXL VAE fp16 出 NaN 黑图）。
6. wd14 打标：自动探测 ONNX 输入名 + CPU 回退。

**执行顺序：** 运行时选 T4 GPU → 逐格执行 → Step 2 上传 `100_cyberboy` 训练集 → Step 7 下载 `.safetensors`。

训练集可用本仓库 `run_demo.sh` 生成（`demo_out/train_data/100_cyberboy/`）。

In [ ]:
# Step 0：环境检查（确认 T4 GPU 与显存）
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('CUDA:', torch.cuda.is_available(), '| 设备:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无')
assert torch.cuda.is_available(), '请先在 菜单-运行时-更改运行时类型 选择 T4 GPU'

In [ ]:
# Step 1：克隆 kohya-ss/sd-scripts（tag v0.8.7）并安装依赖
# 注意：直接克隆 sd-scripts，不要用 bmaltais/kohya_ss（子模块无克隆为空）
# 若与 Colab 预装 torch 冲突，去掉 -b v0.8.7 用最新版
!git clone --depth 1 -b v0.8.7 https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q --upgrade accelerate transformers ftfy tensorboard safetensors albumentations opencv-python

In [ ]:
# Step 2：上传训练集
# 左侧文件面板把 100_cyberboy 文件夹拖到 /content/sd-scripts/train_data/
# 目录结构：train_data/100_cyberboy/*.png + 同名 .txt（100 是 kohya 的 repeats 前缀）
import glob, os
imgs = sorted(glob.glob('train_data/100_cyberboy/*.png'))
print(f'训练图：{len(imgs)} 张')
print('示例：', imgs[:3])
assert imgs, '尚未上传：请把 100_cyberboy 文件夹上传到 /content/sd-scripts/train_data/'

In [ ]:
# Step 3：wd14-tagger 自动打标（自动探测 ONNX 输入名 + CPU 回退）
import glob, os, subprocess

def tag_dir(img_dir):
    imgs = sorted(glob.glob(os.path.join(img_dir, '*.png')))
    imgs += sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
    assert imgs, f'{img_dir} 下没有图片，请先完成 Step 2'
    # 自动探测 ONNX 输入名（不同 onnxruntime 版本 input name 不同，硬编码会 KeyError）
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession('wd14_tagger_model.onnx', providers=['CPUExecutionProvider'])
        print('ONNX 输入名探测:', sess.get_inputs()[0].name)
        del sess
    except Exception:
        print('ONNX 探测失败，走 CPU 回退（脚本默认 provider）')
    cmd = ['python', 'finetune/tag_images_by_wd14_tagger.py',
           '--batch_size', '4',
           '--repo_id', 'SmilingWolf/wd-v1-4-convnext-tagger-v2',
           '--thresh', '0.35', img_dir]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

tag_dir('train_data/100_cyberboy')
print('打标完成：每张图旁生成同名 .txt（触发词 cyberboy 由 keep_tokens=1 保留）')

In [ ]:
# Step 4：单卡非交互式 accelerate 配置（必做，否则 launch 卡在交互问答）
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='fp16')
print('accelerate 配置完成（单卡 fp16）')

In [ ]:
# Step 5：训练脚本（f-string 生成）并启动
# T4 16GB 显存压缩组合：fp16 + AdamW8bit + gradient_checkpointing + cache_latents_to_disk + dim32/alpha16
# OOM 时：network_dim 降到 16 或关 cache_latents；确保 train_batch_size=1
train_script = f'''import subprocess

cmd = [
    "python", "-m", "accelerate.commands.launch",
    "--num_processes=1",
    "--num_machines=1",
    "--mixed_precision=fp16",
    "sdxl_train_network.py",
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
    "--train_data_dir=./train_data",
    "--output_dir=./outputs",
    "--output_name=cyberboy_sdxl",
    "--resolution=1024,1024",
    "--mixed_precision=fp16",
    "--no_half_vae",
    "--optimizer_type=AdamW8bit",
    "--gradient_checkpointing",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--network_module=networks.lora",
    "--network_dim=32",
    "--network_alpha=16",
    "--enable_bucket",
    "--min_bucket_reso=512",
    "--max_bucket_reso=1536",
    "--noise_offset=0.1",
    "--train_batch_size=1",
    "--max_train_epochs=8",
    "--learning_rate=1e-4",
    "--text_encoder_lr=5e-5",
    "--lr_scheduler=cosine",
    "--seed=42",
    "--keep_tokens=1",
    "--save_every_n_epochs=1",
    "--save_precision=fp16",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
'''

# 无 CUDA 时的验证方式：正则提取 f-string 生成的训练脚本 → compile() 静态语法校验
compile(train_script, '<train_script>', 'exec')
print('训练脚本静态语法校验通过')

# 启动训练（约 20~45 分钟 / 8 epochs）
exec(compile(train_script, '<train_script>', 'exec'))

In [ ]:
# Step 6：确认产物（.safetensors 0KB 说明训练异常终止，需至少跑完 1 个 epoch）
import glob, os
outs = sorted(glob.glob('outputs/*.safetensors'))
for o in outs:
    print(f'{o}  {os.path.getsize(o)} bytes')
if not outs:
    print('尚未产出 .safetensors：训练还在进行或未启动，回到 Step 5 检查日志')

In [ ]:
# Step 7：下载 LoRA 产物（.safetensors 50~100MB）
import glob, os
from google.colab import files
for o in sorted(glob.glob('outputs/*.safetensors')):
    if os.path.getsize(o) > 0:
        files.download(o)
        print(f'已触发下载：{o}')

# 下载后：
# 本地推理  ./venv-infer/bin/python inference.py -s studio --lora ./cyberboy_sdxl.safetensors --compare --ref 本人照片目录
# WebUI/ComfyUI  把 .safetensors 放进 models/Lora/，套用 prompts_generated.md 的 Prompt（触发词 cyberboy 放最前）